# Mistral-7B Finance QLoRA — A100 training (zero-manual-edit)

**How to run:** Runtime → Change runtime type → **A100 GPU**. Then confirm the two secrets in the next cell and **Runtime → Run all**. No cell editing, no path fiddling.

Checkpoints are written to your Google Drive every `save_steps`, so a disconnect costs at most that many steps. Re-running resumes from the latest Drive checkpoint automatically.

In [ ]:
# 1) Sanity-check the GPU. This project is built for an Ampere A100 (SM 8.0),
#    which is what enables Flash Attention 2 + bf16.
import torch
assert torch.cuda.is_available(), 'Select an A100 GPU runtime first.'
cc = torch.cuda.get_device_capability()
print('GPU:', torch.cuda.get_device_name(0), '| compute capability:', cc)
assert cc[0] >= 8, 'This config targets Ampere+ (A100). FA2/bf16 need SM>=8.0.'

In [ ]:
# 2) Get the code. Replace the URL with your fork if needed.
import os
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/your-username/llm-finetuning-qlora.git')
if not os.path.exists('llm-finetuning-qlora'):
    !git clone $REPO_URL llm-finetuning-qlora
%cd llm-finetuning-qlora

In [ ]:
# 3) Install pinned deps WITHOUT touching Colab's CUDA torch.
#    We pin torch to whatever Colab already ships (constraints file) so
#    nothing in requirements.txt can upgrade/downgrade it. torch is NOT in
#    requirements.txt anymore (see comment there).
import torch
with open('/tmp/constraints.txt', 'w') as f:
    f.write(f'torch=={torch.__version__}\n')
!pip install -q -r requirements.txt -c /tmp/constraints.txt
# flash-attn: pinned, built against the runtime torch via --no-build-isolation.
# Prefer a prebuilt wheel matching Colab's torch/CUDA/Python; this pin is the
# source of truth (see requirements.txt).
!pip install -q flash-attn==2.7.4.post1 --no-build-isolation -c /tmp/constraints.txt

In [ ]:
# 3b) HARD CHECK: confirm torch is STILL a CUDA build after the installs.
#     If anything replaced it with a CPU wheel, abort now -- training on CPU
#     by accident would silently waste paid compute units.
import torch
print('torch:', torch.__version__, '| torch.version.cuda:', torch.version.cuda,
      '| cuda available:', torch.cuda.is_available())
assert torch.version.cuda is not None, (
    'torch is a CPU build! Something reinstalled torch. Do NOT train -- '
    'restart runtime and reinstall without upgrading torch.')
assert torch.cuda.is_available(), 'CUDA torch present but no GPU visible -- check runtime.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 4) Secrets. Prefer Colab Secrets (key icon, left bar): add HF_TOKEN and
#    WANDB_API_KEY there. Falls back to a prompt if not set.
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    try:
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    except Exception:
        print('No WANDB_API_KEY secret — W&B logging will be disabled.')
except Exception:
    from getpass import getpass
    os.environ['HF_TOKEN'] = getpass('HF_TOKEN (needs Mistral-7B license access): ')
    os.environ['WANDB_API_KEY'] = getpass('WANDB_API_KEY (blank to skip): ')
from huggingface_hub import login
login(os.environ['HF_TOKEN'])

In [ ]:
# 5) Mount Drive and pull any existing checkpoint so we resume, not restart.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_CKPT = '/content/drive/MyDrive/llm_qlora_ckpts/mistral7b-finance-qlora'
LOCAL_OUT = 'outputs/mistral7b-finance-qlora'
!python scripts/pull_checkpoint.py --src "$DRIVE_CKPT" --dst "$LOCAL_OUT" || true

In [ ]:
# 6) Confirm the runtime guards picked FA2 + bf16 + 4-bit on this A100.
from src.config import describe_environment
print(describe_environment())  # expect flash_attention_2 / uses_4bit_quant True / training_bf16 True

In [ ]:
# 7) Prepare data (idempotent; cached). Real dataset = gbharti/finance-alpaca.
from src.config import load_settings
from src.data.prepare import prepare_dataset
settings = load_settings()  # smoke_test=False -> real Mistral-7B + 50k subsample
print(prepare_dataset(settings))

In [ ]:
# 8) Train. Checkpoints to Drive every save_steps; auto-resume if present.
#    Point output_dir straight at Drive so checkpoints survive disconnects.
from dataclasses import replace
from src.train.sft import train
settings = replace(settings, output_dir=DRIVE_CKPT, resume_from_checkpoint=True,
                   push_to_hub=True, hub_model_id='your-username/mistral7b-finance-qlora')
metrics = train(settings)
print('REAL metrics from this A100 run:', metrics)  # peak_vram_gb etc. are measured

In [ ]:
# 9) Push the final adapter to the Hub.
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(settings.hub_model_id, exist_ok=True)
api.upload_folder(folder_path=metrics['adapter_dir'], repo_id=settings.hub_model_id)
print('Pushed adapter to', settings.hub_model_id)